In [1]:
import os
import torch
import pandas as pd
from transformers import DistilBertTokenizerFast
from torch.utils.data import DataLoader
from optional_fine_tune import BertTokenClassification, DfToDataset

D:\Codding\Education\NLP\Tweet Sentiment Extraction\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используем устройство: {device}")

MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

Используем устройство: cuda


In [3]:
print("Загрузка весов моделей с диска...")
models = []

for fold in range(5):
    model = BertTokenClassification(MODEL_NAME)
    model_path = f"models/best_bert_fold_{fold + 1}.pt"

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Файл весов {model_path} не найден! Убедись, что он лежит в папке со скриптом.")

    weights = torch.load(model_path, map_location=device)
    model.load_state_dict(weights)
    model.to(device)
    model.eval()  # Отключаем Dropout!
    models.append(model)
print("Все 5 моделей успешно загружены в память.")

Загрузка весов моделей с диска...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12453.03it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\delux\AppData\Local\Temp\ipykernel_8820\1687977759.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, t

Все 5 моделей успешно загружены в память.


In [4]:
test = pd.read_csv("data/test.csv")
test_dataset = DfToDataset(test, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [5]:
print("Запуск предсказания на тестовых данных...")
all_blend_preds = []

predictions = []
real_values = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        offsets_mapping = batch["offset_mapping"].numpy()
        texts = batch["text"]
        sentiments = batch["sentiment"]

        start_logits_avg = 0
        end_logits_avg = 0

        for model in models:
            start_logits, end_logits = model(input_ids=input_ids, attention_mask=attention_mask)
            start_logits_avg += start_logits
            end_logits_avg += end_logits

        start_logits_avg /= len(models)
        end_logits_avg /= len(models)

        start_preds = torch.argmax(start_logits_avg, dim=1).cpu().numpy()
        end_preds = torch.argmax(end_logits_avg, dim=1).cpu().numpy()

        for i in range(len(texts)):
            if sentiments[i] == 'neutral':
                predicted_str = texts[i]
            else:
                start_idx = start_preds[i]
                end_idx = end_preds[i]

                if start_idx > end_idx:
                    predicted_str = texts[i]
                else:
                    char_start = offsets_mapping[i][start_idx][0]
                    char_end = offsets_mapping[i][end_idx][1]
                    predicted_str = texts[i][char_start:char_end]

            predictions.append(predicted_str)

submission = pd.DataFrame({
    "textID": test["textID"],
    "selected_text": predictions
})

submission.to_csv("results/submission_ensemble.csv", index=False)
print("Файл submission_ensemble.csv успешно создан и готов к отправке!")

Запуск предсказания на тестовых данных...
Файл submission_ensemble.csv успешно создан и готов к отправке!
